In [2]:
import numpy as np
import qutip as qt
from scipy.linalg import expm
from qutip import negativity

from scipy.linalg import logm, eigvalsh
import matplotlib.pyplot as plt
from itertools import permutations

In [3]:
def partial_trace(rho, keep, dims):
    n = len(dims)
    rho_tensor = rho.reshape(dims + dims)
    trace_out = [i for i in range(n) if i not in keep]
    for sys in sorted(trace_out, reverse=True):
        n_current = len(rho_tensor.shape) // 2
        rho_tensor = np.trace(rho_tensor, axis1=sys, axis2=sys + n_current)
    d_keep = int(np.prod([dims[i] for i in keep]))
    return rho_tensor.reshape(d_keep, d_keep)

def von_neumann_entropy(rho, base=2, tol=1e-12):
    eigvals = eigvalsh(rho)
    eigvals = eigvals[eigvals > tol]
    return -np.sum(eigvals * np.log(eigvals) / np.log(base))

def jordan_product(A, B):
    return 0.5 * (A @ B + B @ A)

def jordan_associator(A, B, C):
    return jordan_product(jordan_product(A, B), C) - jordan_product(A, jordan_product(B, C))

Heisenberg-inspired family

In [46]:
# =========================
# Basic single-qubit operators
# =========================
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
ket_plus = (ket0 + ket1) / np.sqrt(2)

In [47]:
# =========================
# Utility helpers
# =========================
def kron_all(op_list):
    out = np.array([[1.0 + 0j]])
    for op in op_list:
        out = np.kron(out, op)
    return out

def embed_two_body(n_qubits, i, j, op_i, op_j):
    ops = [I2] * n_qubits
    ops[i] = op_i
    ops[j] = op_j
    return kron_all(ops)

def xyz_edge_hamiltonian(n_qubits, i, j, Jx, Jy, Jz):
    return (
        Jx * embed_two_body(n_qubits, i, j, sx, sx)
        + Jy * embed_two_body(n_qubits, i, j, sy, sy)
        + Jz * embed_two_body(n_qubits, i, j, sz, sz)
    )

def commutator(A, B):
    return A @ B - B @ A

def mutual_info(rho_XY, keep_X, keep_Y, dims):
    rho_X = partial_trace(rho_XY, keep=keep_X, dims=dims)
    rho_Y = partial_trace(rho_XY, keep=keep_Y, dims=dims)
    return (
        von_neumann_entropy(rho_X)
        + von_neumann_entropy(rho_Y)
        - von_neumann_entropy(rho_XY)
    )

def I3_of_rhoABC(rho_ABC):
    rho_A  = partial_trace(rho_ABC, keep=[0], dims=[2, 2, 2])
    rho_B  = partial_trace(rho_ABC, keep=[1], dims=[2, 2, 2])
    rho_C  = partial_trace(rho_ABC, keep=[2], dims=[2, 2, 2])
    rho_AB = partial_trace(rho_ABC, keep=[0, 1], dims=[2, 2, 2])
    rho_AC = partial_trace(rho_ABC, keep=[0, 2], dims=[2, 2, 2])
    rho_BC = partial_trace(rho_ABC, keep=[1, 2], dims=[2, 2, 2])

    I3 = (
        von_neumann_entropy(rho_A)
        + von_neumann_entropy(rho_B)
        + von_neumann_entropy(rho_C)
        - von_neumann_entropy(rho_AB)
        - von_neumann_entropy(rho_AC)
        - von_neumann_entropy(rho_BC)
        + von_neumann_entropy(rho_ABC)
    )
    return float(np.real_if_close(I3))

def mediator_metrics(rho_ABC, H_AB_3, H_BC_3, H_AC_3):
    # mediator channels
    A_AB = jordan_associator(H_BC_3, H_AB_3, H_AC_3)  # mediator = H_AB
    A_BC = jordan_associator(H_AB_3, H_BC_3, H_AC_3)  # mediator = H_BC
    A_AC = jordan_associator(H_AB_3, H_AC_3, H_BC_3)  # mediator = H_AC

    # bare mediator scores
    j_AB = np.linalg.norm(A_AB, 'fro')
    j_BC = np.linalg.norm(A_BC, 'fro')
    j_AC = np.linalg.norm(A_AC, 'fro')
    J_tot = np.sqrt(j_AB**2 + j_BC**2 + j_AC**2)

    # state-resolved activations
    m_AB = float(np.real_if_close(np.trace(rho_ABC @ A_AB)))
    m_BC = float(np.real_if_close(np.trace(rho_ABC @ A_BC)))
    m_AC = float(np.real_if_close(np.trace(rho_ABC @ A_AC)))
    J_rho_exp = np.sqrt(np.abs(m_AB)**2 + np.abs(m_BC)**2 + np.abs(m_AC)**2)

    return {
        "A_AB": A_AB, "A_BC": A_BC, "A_AC": A_AC,
        "j_AB": j_AB, "j_BC": j_BC, "j_AC": j_AC,
        "J_tot": J_tot,
        "m_AB": m_AB, "m_BC": m_BC, "m_AC": m_AC,
        "J_rho_exp": J_rho_exp,
    }

In [48]:
# =========================
# Fixed internal family form (ABC), different coefficients on 3 edges
# =========================
J_AB = (1.00, 0.70, 0.30)
J_BC = (0.80, 1.00, 0.50)
J_AC = (0.60, 0.40, 1.00)

# Weak environment coupling shape (C-D edge), overall scale set by gD
J_CD_shape = (1.00, 0.80, 0.60)

# =========================
# 3-qubit internal probes on ABC (for associator after tracing out D)
# qubit order on ABC = A(0), B(1), C(2)
# =========================
H_AB_3 = xyz_edge_hamiltonian(3, 0, 1, *J_AB)
H_BC_3 = xyz_edge_hamiltonian(3, 1, 2, *J_BC)
H_AC_3 = xyz_edge_hamiltonian(3, 0, 2, *J_AC)

# =========================
# Initial state: |+ + + 0>
# qubit order on ABCD = A(0), B(1), C(2), D(3)
# =========================
psi0 = kron_all([ket_plus, ket_plus, ket_plus, ket0]).reshape(-1)

In [49]:
def run_case(gD=0.15, t=1.0, verbose=True):
    # 4-qubit Hamiltonian
    H_AB_4 = xyz_edge_hamiltonian(4, 0, 1, *J_AB)
    H_BC_4 = xyz_edge_hamiltonian(4, 1, 2, *J_BC)
    H_AC_4 = xyz_edge_hamiltonian(4, 0, 2, *J_AC)

    H_CD_4 = gD * xyz_edge_hamiltonian(4, 2, 3, *J_CD_shape)

    H_tot = H_AB_4 + H_BC_4 + H_AC_4 + H_CD_4

    # time evolution
    U = expm(-1j * t * H_tot)
    psi_t = U @ psi0
    rho4 = np.outer(psi_t, psi_t.conj())

    # reduced mixed state on ABC
    rho_ABC = partial_trace(rho4, keep=[0, 1, 2], dims=[2, 2, 2, 2])

    # state diagnostics
    evals = np.sort(np.real_if_close(eigvalsh(rho_ABC)))[::-1]
    S_ABC = von_neumann_entropy(rho_ABC)
    I3 = I3_of_rhoABC(rho_ABC)
    I3_minus = max(0.0, -I3)

    # pairwise mutual infos on ABC
    rho_AB = partial_trace(rho_ABC, keep=[0, 1], dims=[2, 2, 2])
    rho_AC = partial_trace(rho_ABC, keep=[0, 2], dims=[2, 2, 2])
    rho_BC = partial_trace(rho_ABC, keep=[1, 2], dims=[2, 2, 2])

    I_AB = mutual_info(rho_AB, keep_X=[0], keep_Y=[1], dims=[2, 2])
    I_AC = mutual_info(rho_AC, keep_X=[0], keep_Y=[1], dims=[2, 2])
    I_BC = mutual_info(rho_BC, keep_X=[0], keep_Y=[1], dims=[2, 2])

    # associator metrics
    med = mediator_metrics(rho_ABC, H_AB_3, H_BC_3, H_AC_3)

    if verbose:
        print(f"\n=== Case: gD = {gD:.2f}, t = {t:.2f} ===")
        print("Top eigenvalues of rho_ABC:", np.round(evals[:4], 6))
        print(f"S(rho_ABC) = {S_ABC:.6f}")
        print(f"I3 = {I3:.6f}, I3_minus = {I3_minus:.6f}")
        print(f"I(A:B) = {I_AB:.6f}, I(A:C) = {I_AC:.6f}, I(B:C) = {I_BC:.6f}")
        print(f"j_AB = {med['j_AB']:.6f}, j_BC = {med['j_BC']:.6f}, j_AC = {med['j_AC']:.6f}, J_tot = {med['J_tot']:.6f}")
        print(f"m_AB = {med['m_AB']:.6f}, m_BC = {med['m_BC']:.6f}, m_AC = {med['m_AC']:.6f}, J_rho_exp = {med['J_rho_exp']:.6f}")

    return {
        "gD": gD,
        "t": t,
        "rho_ABC": rho_ABC,
        "eigvals": evals,
        "S_ABC": S_ABC,
        "I3": I3,
        "I3_minus": I3_minus,
        "I_AB": I_AB,
        "I_AC": I_AC,
        "I_BC": I_BC,
        **med,
    }

In [50]:
# =========================
# Run a few simple weak-coupling cases (not a scan)
# =========================
res_10 = run_case(gD=0.10, t=1.0)
res_15 = run_case(gD=0.15, t=1.0)
res_20 = run_case(gD=0.20, t=1.0)


=== Case: gD = 0.10, t = 1.00 ===
Top eigenvalues of rho_ABC: [0.995569 0.004431 0.       0.      ]
S(rho_ABC) = 0.041020
I3 = -0.005044, I3_minus = 0.005044
I(A:B) = 0.179221, I(A:C) = 0.596624, I(B:C) = 0.223434
j_AB = 4.965059, j_BC = 4.975646, j_AC = 4.328055, J_tot = 8.254752
m_AB = -0.376422, m_BC = -0.355793, m_AC = 0.020630, J_rho_exp = 0.518370

=== Case: gD = 0.15, t = 1.00 ===
Top eigenvalues of rho_ABC: [0.990398 0.009602 0.       0.      ]
S(rho_ABC) = 0.078141
I3 = -0.012507, I3_minus = 0.012507
I(A:B) = 0.175437, I(A:C) = 0.580340, I(B:C) = 0.219706
j_AB = 4.965059, j_BC = 4.975646, j_AC = 4.328055, J_tot = 8.254752
m_AB = -0.373569, m_BC = -0.354232, m_AC = 0.019338, J_rho_exp = 0.515178

=== Case: gD = 0.20, t = 1.00 ===
Top eigenvalues of rho_ABC: [0.983684 0.016316 0.       0.      ]
S(rho_ABC) = 0.120220
I3 = -0.023256, I3_minus = 0.023256
I(A:B) = 0.171848, I(A:C) = 0.560767, I(B:C) = 0.217446
j_AB = 4.965059, j_BC = 4.975646, j_AC = 4.328055, J_tot = 8.254752
m_A

Fix the energy, scan the angle

In [58]:
# =========================================================
# Assumed already defined in your notebook:
# partial_trace, von_neumann_entropy, jordan_product, jordan_associator
# =========================================================

# ---------- basic single-qubit operators ----------
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
ket_plus = (ket0 + ket1) / np.sqrt(2)

# ---------- helpers ----------
def kron_all(op_list):
    out = np.array([[1.0 + 0j]])
    for op in op_list:
        out = np.kron(out, op)
    return out

def embed_two_body(n_qubits, i, j, op_i, op_j):
    ops = [I2] * n_qubits
    ops[i] = op_i
    ops[j] = op_j
    return kron_all(ops)

def xyz_edge_hamiltonian(n_qubits, i, j, Jx, Jy, Jz):
    return (
        Jx * embed_two_body(n_qubits, i, j, sx, sx)
        + Jy * embed_two_body(n_qubits, i, j, sy, sy)
        + Jz * embed_two_body(n_qubits, i, j, sz, sz)
    )

def to_real_scalar(x, tol=1e-10):
    x = np.real_if_close(x, tol=1000)
    if np.iscomplexobj(x):
        if abs(np.imag(x)) < tol:
            return float(np.real(x))
        raise ValueError(f"Expected nearly real scalar, got {x}")
    return float(x)

def commutator(A, B):
    return A @ B - B @ A

def mutual_info_2q(rho_2q):
    rho_1 = partial_trace(rho_2q, keep=[0], dims=[2, 2])
    rho_2 = partial_trace(rho_2q, keep=[1], dims=[2, 2])
    return (
        von_neumann_entropy(rho_1)
        + von_neumann_entropy(rho_2)
        - von_neumann_entropy(rho_2q)
    )

def I3_of_rhoABC(rho_ABC):
    rho_A  = partial_trace(rho_ABC, keep=[0], dims=[2, 2, 2])
    rho_B  = partial_trace(rho_ABC, keep=[1], dims=[2, 2, 2])
    rho_C  = partial_trace(rho_ABC, keep=[2], dims=[2, 2, 2])
    rho_AB = partial_trace(rho_ABC, keep=[0, 1], dims=[2, 2, 2])
    rho_AC = partial_trace(rho_ABC, keep=[0, 2], dims=[2, 2, 2])
    rho_BC = partial_trace(rho_ABC, keep=[1, 2], dims=[2, 2, 2])

    I3 = (
        von_neumann_entropy(rho_A)
        + von_neumann_entropy(rho_B)
        + von_neumann_entropy(rho_C)
        - von_neumann_entropy(rho_AB)
        - von_neumann_entropy(rho_AC)
        - von_neumann_entropy(rho_BC)
        + von_neumann_entropy(rho_ABC)
    )
    return to_real_scalar(I3)

def mediator_metrics(rho_ABC, H_AB_3, H_BC_3, H_AC_3):
    # ordered associators, middle operator is the mediator
    A_AB = jordan_associator(H_BC_3, H_AB_3, H_AC_3)  # mediator = H_AB
    A_BC = jordan_associator(H_AB_3, H_BC_3, H_AC_3)  # mediator = H_BC
    A_AC = jordan_associator(H_AB_3, H_AC_3, H_BC_3)  # mediator = H_AC

    # bare mediator scores
    j_AB = np.linalg.norm(A_AB, 'fro')
    j_BC = np.linalg.norm(A_BC, 'fro')
    j_AC = np.linalg.norm(A_AC, 'fro')
    J_tot = np.sqrt(j_AB**2 + j_BC**2 + j_AC**2)

    # state-resolved activations
    m_AB = to_real_scalar(np.trace(rho_ABC @ A_AB))
    m_BC = to_real_scalar(np.trace(rho_ABC @ A_BC))
    m_AC = to_real_scalar(np.trace(rho_ABC @ A_AC))
    J_rho_exp = np.sqrt(np.abs(m_AB)**2 + np.abs(m_BC)**2 + np.abs(m_AC)**2)

    return {
        "A_AB": A_AB, "A_BC": A_BC, "A_AC": A_AC,
        "j_AB": j_AB, "j_BC": j_BC, "j_AC": j_AC,
        "J_tot": J_tot,
        "m_AB": m_AB, "m_BC": m_BC, "m_AC": m_AC,
        "J_rho_exp": J_rho_exp,
    }

# =========================================================
# Fixed-norm family design
# =========================================================

# total norm for each internal edge
R = np.sqrt(3.0)

# edge-specific shape templates (same family form, different local weights)
shape_AB = np.array([1.00, 0.70, 0.30], dtype=float)
shape_BC = np.array([0.80, 1.00, 0.50], dtype=float)
shape_AC = np.array([0.60, 0.40, 1.00], dtype=float)

# D only hangs on C; environment coupling is weak
shape_CD = np.array([1.00, 0.80, 0.60], dtype=float)

def unit_direction(theta, phi):
    """
    Spherical direction on coefficient sphere:
      n = (sinθ cosφ, sinθ sinφ, cosθ)
    """
    return np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta)
    ], dtype=float)

def normalized_edge_coeffs(theta, phi, shape, R=np.sqrt(3.0), eps=1e-12):
    """
    Build edge-specific XYZ coefficients:
      1) multiply global direction n(theta,phi) elementwise by edge shape
      2) renormalize to fixed norm R

    This keeps total coupling strength fixed while allowing anisotropy direction to vary.
    """
    vec = unit_direction(theta, phi) * shape
    norm = np.linalg.norm(vec)
    if norm < eps:
        raise ValueError("Degenerate coefficient vector; choose different theta/phi.")
    return R * vec / norm

# =========================================================
# Initial state: |+ + + 0>
# qubit order on ABCD = A(0), B(1), C(2), D(3)
# =========================================================
psi0 = kron_all([ket_plus, ket_plus, ket_plus, ket0]).reshape(-1)

# =========================================================
# Single-run evaluator at fixed (theta, phi, gD, t)
# =========================================================
def run_fixed_norm_case(theta, phi, gD=0.20, t=1.0, verbose=True):
    # ---- internal edge coefficients (fixed norm R) ----
    J_AB = normalized_edge_coeffs(theta, phi, shape_AB, R=R)
    J_BC = normalized_edge_coeffs(theta, phi, shape_BC, R=R)
    J_AC = normalized_edge_coeffs(theta, phi, shape_AC, R=R)

    # ---- weak environment coupling on C-D ----
    # same angular direction, fixed shape, norm = gD * R
    J_CD = normalized_edge_coeffs(theta, phi, shape_CD, R=gD * R)

    # ---- 4-qubit Hamiltonian for state generation ----
    H_AB_4 = xyz_edge_hamiltonian(4, 0, 1, *J_AB)
    H_BC_4 = xyz_edge_hamiltonian(4, 1, 2, *J_BC)
    H_AC_4 = xyz_edge_hamiltonian(4, 0, 2, *J_AC)
    H_CD_4 = xyz_edge_hamiltonian(4, 2, 3, *J_CD)

    H_tot_4 = H_AB_4 + H_BC_4 + H_AC_4 + H_CD_4

    # ---- evolve pure state ----
    U = expm(-1j * t * H_tot_4)
    psi_t = U @ psi0
    rho4 = np.outer(psi_t, psi_t.conj())

    # ---- trace out D -> rho_ABC ----
    rho_ABC = partial_trace(rho4, keep=[0, 1, 2], dims=[2, 2, 2, 2])

    # ---- pairwise reduced states ----
    rho_AB = partial_trace(rho_ABC, keep=[0, 1], dims=[2, 2, 2])
    rho_AC = partial_trace(rho_ABC, keep=[0, 2], dims=[2, 2, 2])
    rho_BC = partial_trace(rho_ABC, keep=[1, 2], dims=[2, 2, 2])

    I_AB = mutual_info_2q(rho_AB)
    I_AC = mutual_info_2q(rho_AC)
    I_BC = mutual_info_2q(rho_BC)
    P_pair = I_AB + I_AC + I_BC

    # ---- TMI ----
    I3 = I3_of_rhoABC(rho_ABC)
    I3_minus = max(0.0, -I3)

    # ---- reduced-state entropy (environment-induced mixedness) ----
    S_ABC = von_neumann_entropy(rho_ABC)
    eigvals = np.sort(np.real_if_close(np.linalg.eigvalsh(rho_ABC)))[::-1]

    # ---- 3-qubit internal Hamiltonians as probes ----
    H_AB_3 = xyz_edge_hamiltonian(3, 0, 1, *J_AB)
    H_BC_3 = xyz_edge_hamiltonian(3, 1, 2, *J_BC)
    H_AC_3 = xyz_edge_hamiltonian(3, 0, 2, *J_AC)

    med = mediator_metrics(rho_ABC, H_AB_3, H_BC_3, H_AC_3)
    eta = med["J_rho_exp"] / med["J_tot"] if med["J_tot"] > 1e-12 else np.nan

    out = {
        "theta": theta,
        "phi": phi,
        "gD": gD,
        "t": t,
        "J_AB": J_AB, "J_BC": J_BC, "J_AC": J_AC, "J_CD": J_CD,
        "rho_ABC": rho_ABC,
        "eigvals": eigvals,
        "S_ABC": S_ABC,
        "I3": I3,
        "I3_minus": I3_minus,
        "I_AB": I_AB, "I_AC": I_AC, "I_BC": I_BC,
        "P_pair": P_pair,
        "eta": eta,
        **med,
    }

    if verbose:
        print(f"\n=== theta={theta:.3f}, phi={phi:.3f}, gD={gD:.2f}, t={t:.2f} ===")
        print("J_AB =", np.round(J_AB, 4))
        print("J_BC =", np.round(J_BC, 4))
        print("J_AC =", np.round(J_AC, 4))
        print("J_CD =", np.round(J_CD, 4))
        print("Top eigvals(rho_ABC) =", np.round(eigvals[:4], 6))
        print(f"S(rho_ABC) = {S_ABC:.6f}")
        print(f"I3 = {I3:.6f}, I3_minus = {I3_minus:.6f}")
        print(f"I(A:B) = {I_AB:.6f}, I(A:C) = {I_AC:.6f}, I(B:C) = {I_BC:.6f}, P_pair = {P_pair:.6f}")
        print(f"j_AB = {med['j_AB']:.6f}, j_BC = {med['j_BC']:.6f}, j_AC = {med['j_AC']:.6f}, J_tot = {med['J_tot']:.6f}")
        print(f"m_AB = {med['m_AB']:.6f}, m_BC = {med['m_BC']:.6f}, m_AC = {med['m_AC']:.6f}, J_rho_exp = {med['J_rho_exp']:.6f}, eta = {eta:.6f}")

    return out

# =========================================================
# Compact scan over (theta, phi) at fixed gD and t
# =========================================================
def scan_fixed_norm(theta_vals, phi_vals, gD=0.20, t=1.0, verbose_every=0):
    results = []
    counter = 0
    for th in theta_vals:
        for ph in phi_vals:
            counter += 1
            verbose = (verbose_every > 0 and counter % verbose_every == 0)
            try:
                res = run_fixed_norm_case(th, ph, gD=gD, t=t, verbose=verbose)
                results.append(res)
            except ValueError:
                # skip degenerate direction
                continue
    return results

# =========================================================
# Optional summary helpers
# =========================================================
def summarize_scan(results, top_k=10):
    if len(results) == 0:
        print("No results.")
        return

    I3m = np.array([r["I3_minus"] for r in results], dtype=float)
    Jr  = np.array([r["J_rho_exp"] for r in results], dtype=float)
    Jt  = np.array([r["J_tot"] for r in results], dtype=float)
    Eta = np.array([r["eta"] for r in results], dtype=float)
    Pp  = np.array([r["P_pair"] for r in results], dtype=float)

    def safe_corr(x, y):
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            return np.nan
        return np.corrcoef(x, y)[0, 1]

    corr_Jr_I3m  = safe_corr(Jr, I3m)
    corr_Jt_I3m  = safe_corr(Jt, I3m)
    corr_Eta_I3m = safe_corr(Eta, I3m)
    corr_Pp_I3m  = safe_corr(Pp, I3m)

    print("\n=== Scan summary ===")
    print(f"N = {len(results)}")
    print(f"Pearson corr(J_rho_exp, I3_minus) = {corr_Jr_I3m:.6f}")
    print(f"Pearson corr(J_tot,     I3_minus) = {corr_Jt_I3m:.6f}")
    print(f"Pearson corr(eta,       I3_minus) = {corr_Eta_I3m:.6f}")
    print(f"Pearson corr(P_pair,    I3_minus) = {corr_Pp_I3m:.6f}")

    idx = np.argsort(-I3m)[:top_k]
    print(f"\nTop {top_k} points by I3_minus:")
    for k in idx:
        r = results[k]
        print(
            f"theta={r['theta']:.3f}, phi={r['phi']:.3f}, "
            f"I3_minus={r['I3_minus']:.6f}, "
            f"J_rho_exp={r['J_rho_exp']:.6f}, "
            f"J_tot={r['J_tot']:.6f}, "
            f"eta={r['eta']:.6f}, "
            f"P_pair={r['P_pair']:.6f}"
        )

In [60]:
def summarize_scan_numeric(results):
    if len(results) == 0:
        return {
            "N": 0,
            "corr_J_rho_exp": np.nan,
            "corr_J_tot": np.nan,
            "corr_eta": np.nan,
            "corr_P_pair": np.nan,
            "max_I3_minus": np.nan,
            "mean_I3_minus": np.nan,
        }

    I3m = np.array([r["I3_minus"] for r in results], dtype=float)
    Jr  = np.array([r["J_rho_exp"] for r in results], dtype=float)
    Jt  = np.array([r["J_tot"] for r in results], dtype=float)
    Eta = np.array([r["eta"] for r in results], dtype=float)
    Pp  = np.array([r["P_pair"] for r in results], dtype=float)

    def safe_corr(x, y):
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            return np.nan
        return np.corrcoef(x, y)[0, 1]

    return {
        "N": len(results),
        "corr_J_rho_exp": safe_corr(Jr, I3m),
        "corr_J_tot": safe_corr(Jt, I3m),
        "corr_eta": safe_corr(Eta, I3m),
        "corr_P_pair": safe_corr(Pp, I3m),
        "max_I3_minus": float(np.max(I3m)),
        "mean_I3_minus": float(np.mean(I3m)),
    }

def scan_over_time(theta_vals, phi_vals, t_vals, gD=0.20):
    all_summaries = []
    for t in t_vals:
        results_t = scan_fixed_norm(theta_vals, phi_vals, gD=gD, t=t)
        summary_t = summarize_scan_numeric(results_t)
        summary_t["t"] = t
        all_summaries.append(summary_t)
    return all_summaries

In [61]:
theta_vals = np.linspace(0.1, np.pi/2, 20)
phi_vals   = np.linspace(0.0, np.pi/2, 20)
t_vals     = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.00]

time_summaries = scan_over_time(theta_vals, phi_vals, t_vals, gD=0.20)

for row in time_summaries:
    print(
        f"t={row['t']:.2f} | "
        f"corr(J_rho_exp, I3-)={row['corr_J_rho_exp']:.4f} | "
        f"corr(J_tot, I3-)={row['corr_J_tot']:.4f} | "
        f"corr(eta, I3-)={row['corr_eta']:.4f} | "
        f"corr(P_pair, I3-)={row['corr_P_pair']:.4f} | "
        f"max(I3-)={row['max_I3_minus']:.4f}"
    )

t=0.05 | corr(J_rho_exp, I3-)=0.8027 | corr(J_tot, I3-)=0.0191 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.2056 | max(I3-)=0.0008
t=0.10 | corr(J_rho_exp, I3-)=0.8084 | corr(J_tot, I3-)=0.0215 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.2175 | max(I3-)=0.0035
t=0.15 | corr(J_rho_exp, I3-)=0.7764 | corr(J_tot, I3-)=0.0202 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.2333 | max(I3-)=0.0089
t=0.20 | corr(J_rho_exp, I3-)=0.6689 | corr(J_tot, I3-)=0.0134 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.2505 | max(I3-)=0.0183
t=0.30 | corr(J_rho_exp, I3-)=0.3458 | corr(J_tot, I3-)=-0.0096 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.2774 | max(I3-)=0.0556
t=0.50 | corr(J_rho_exp, I3-)=0.2593 | corr(J_tot, I3-)=-0.0063 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.2767 | max(I3-)=0.1913
t=1.00 | corr(J_rho_exp, I3-)=0.0966 | corr(J_tot, I3-)=0.5571 | corr(eta, I3-)=nan | corr(P_pair, I3-)=0.5055 | max(I3-)=0.1997


In [62]:
etas = np.array([r["eta"] for r in results], dtype=float)
jtots = np.array([r["J_tot"] for r in results], dtype=float)

print("num eta nan =", np.isnan(etas).sum())
print("num J_tot ~ 0 =", np.sum(np.abs(jtots) < 1e-12))
print("min J_tot =", np.min(jtots))
print("first few bad indices =", np.where(np.isnan(etas))[0][:10])

KeyError: 'eta'

In [53]:
theta_test = np.linspace(0, np.pi/2, 20)
phi_test = np.linspace(0, np.pi/2, 20)
results = scan_fixed_norm(theta_test, phi_test, gD=0.15, t=1.0, verbose_every=50)
summarize_scan(results, top_k=5)


=== theta=0.165, phi=0.744, gD=0.15, t=1.00 ===
J_AB = [0.6373 0.4107 1.5573]
J_BC = [0.3259 0.3751 1.6592]
J_AC = [0.1271 0.078  1.7256]
J_CD = [0.0515 0.0379 0.2518]
Top eigvals(rho_ABC) = [0.998529 0.001471 0.       0.      ]
S(rho_ABC) = 0.015963
I3 = -0.011444, I3_minus = 0.011444
I(A:B) = 1.232985, I(A:C) = 0.343431, I(B:C) = 0.394052, P_pair = 1.970468
j_AB = 4.766285, j_BC = 6.784682, j_AC = 7.595057, J_tot = 11.244300
m_AB = 0.467141, m_BC = 1.408300, m_AC = 0.941158, J_rho_exp = 1.757074

=== theta=0.331, phi=1.571, gD=0.15, t=1.00 ===
J_AB = [0.     1.0829 1.3518]
J_BC = [0.     0.9804 1.4279]
J_AC = [0.     0.2356 1.7159]
J_CD = [0.     0.1081 0.2362]
Top eigvals(rho_ABC) = [0.993122 0.006878 0.       0.      ]
S(rho_ABC) = 0.059298
I3 = -0.057481, I3_minus = 0.057481
I(A:B) = 0.990186, I(A:C) = 0.776411, I(B:C) = 0.681976, P_pair = 2.448573
j_AB = 9.856620, j_BC = 10.574116, j_AC = 11.226981, J_tot = 18.303278
m_AB = -0.034806, m_BC = 0.692219, m_AC = 0.727026, J_rho_exp 

Classic symmetric operator. All Failed!

In [33]:
# Pauli matrices
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

# 三条边的 interaction operators（固定，不依赖态）
# H_AB = np.kron(np.kron(sx, sz), I2)   # σ_x ⊗ σ_z ⊗ I
# H_BC = np.kron(np.kron(I2, sx), sz)   # I ⊗ σ_x ⊗ σ_z
# H_AC = np.kron(np.kron(sz, I2), sx)   # σ_z ⊗ I ⊗ σ_x

# # === 新 probe family：适合当前 GHZ4 -> rho_ABC benchmark ===
# H_AB = np.kron(np.kron(sx, sx), I2)   # X ⊗ X ⊗ I
# H_BC = np.kron(np.kron(I2, sy), sy)   # I ⊗ Y ⊗ Y
# H_AC = np.kron(np.kron(sx, I2), sx)   # X ⊗ I ⊗ X

# ===== asymmetric probe family v1 =====
H_AB = (
    np.kron(np.kron(sx, sz), I2)
    + 0.4 * np.kron(np.kron(sz, sx), I2)
)

H_BC = (
    np.kron(np.kron(I2, sy), sx)
    + 0.7 * np.kron(np.kron(I2, sz), sy)
)

H_AC = (
    np.kron(np.kron(sx, I2), sx)
    + 0.5 * np.kron(np.kron(sz, I2), sy)
)

# === 检查 1：对易性 ===
comm_AB_BC = H_AB @ H_BC - H_BC @ H_AB
comm_BC_AC = H_BC @ H_AC - H_AC @ H_BC
comm_AB_AC = H_AB @ H_AC - H_AC @ H_AB

print("=== 对易性检查 ===")
print(f"[H_AB, H_BC] = 0?  {np.allclose(comm_AB_BC, 0)}   ‖[·,·]‖ = {np.linalg.norm(comm_AB_BC, 'fro'):.6f}")
print(f"[H_BC, H_AC] = 0?  {np.allclose(comm_BC_AC, 0)}   ‖[·,·]‖ = {np.linalg.norm(comm_BC_AC, 'fro'):.6f}")
print(f"[H_AB, H_AC] = 0?  {np.allclose(comm_AB_AC, 0)}   ‖[·,·]‖ = {np.linalg.norm(comm_AB_AC, 'fro'):.6f}")

# === 检查 2：裸结合子（不涉及态） ===
assoc_bare = jordan_associator(H_AB, H_BC, H_AC)
print(f"\n=== 裸结合子 [H_AB, H_BC, H_AC]∘ ===")
print(f"‖assoc‖_F = {np.linalg.norm(assoc_bare, 'fro'):.6f}")
print(f"Tr[assoc] = {np.trace(assoc_bare):.6f}")
print(f"Hermitian? {np.allclose(assoc_bare, assoc_bare.conj().T)}")

# === 检查 3：所有 6 种排列的结合子范数 ===
print(f"\n=== 排列结合子范数 ===")
ops = [H_AB, H_BC, H_AC]
labels = ["H_AB", "H_BC", "H_AC"]
for p in permutations(range(3)):
    norm = np.linalg.norm(jordan_associator(ops[p[0]], ops[p[1]], ops[p[2]]), 'fro')
    print(f"  [{labels[p[0]]}, {labels[p[1]]}, {labels[p[2]]}]∘  ‖·‖ = {norm:.6f}")

=== 对易性检查 ===
[H_AB, H_BC] = 0?  False   ‖[·,·]‖ = 6.295141
[H_BC, H_AC] = 0?  False   ‖[·,·]‖ = 4.866210
[H_AB, H_AC] = 0?  False   ‖[·,·]‖ = 3.622154

=== 裸结合子 [H_AB, H_BC, H_AC]∘ ===
‖assoc‖_F = 1.131371
Tr[assoc] = 0.000000+0.000000j
Hermitian? True

=== 排列结合子范数 ===
  [H_AB, H_BC, H_AC]∘  ‖·‖ = 1.131371
  [H_AB, H_AC, H_BC]∘  ‖·‖ = 1.264911
  [H_BC, H_AB, H_AC]∘  ‖·‖ = 0.565685
  [H_BC, H_AC, H_AB]∘  ‖·‖ = 1.264911
  [H_AC, H_AB, H_BC]∘  ‖·‖ = 0.565685
  [H_AC, H_BC, H_AB]∘  ‖·‖ = 1.131371


In [34]:
def commutator(A, B):
    return A @ B - B @ A

A, B, C = H_AB, H_BC, H_AC

AC_comm = commutator(A, C)
nested = commutator(B, AC_comm)
assoc = jordan_associator(A, B, C)

print("‖[A,C]‖_F =", np.linalg.norm(AC_comm, 'fro'))
print("‖[B,[A,C]]‖_F =", np.linalg.norm(nested, 'fro'))
print("‖Assoc(A,B,C)‖_F =", np.linalg.norm(assoc, 'fro'))
print("Check identity:",
      np.allclose(assoc, 0.25 * nested))

‖[A,C]‖_F = 3.622154055254967
‖[B,[A,C]]‖_F = 4.525483399593905
‖Assoc(A,B,C)‖_F = 1.1313708498984762
Check identity: True


In [9]:
def fro_norm(X):
    return np.linalg.norm(X, 'fro')

def mediator_scores_and_tot(H_AB, H_BC, H_AC):
    # 三个 mediator 对应的 associator operators
    A_AB = jordan_associator(H_BC, H_AB, H_AC)  # mediator = H_AB
    A_BC = jordan_associator(H_AB, H_BC, H_AC)  # mediator = H_BC
    A_AC = jordan_associator(H_AB, H_AC, H_BC)  # mediator = H_AC

    # 三个裸 mediator scores
    j_AB = fro_norm(A_AB)
    j_BC = fro_norm(A_BC)
    j_AC = fro_norm(A_AC)

    # 总裸强度
    J_tot = np.sqrt(j_AB**2 + j_BC**2 + j_AC**2)

    return {
        "A_AB": A_AB,
        "A_BC": A_BC,
        "A_AC": A_AC,
        "j_AB": j_AB,
        "j_BC": j_BC,
        "j_AC": j_AC,
        "J_tot": J_tot,
    }

def state_activated_Jrho(rho_ABC, A_AB, A_BC, A_AC):
    # 保证是实数
    val = np.trace(
        rho_ABC @ (
            A_AB.conj().T @ A_AB +
            A_BC.conj().T @ A_BC +
            A_AC.conj().T @ A_AC
        )
    )
    return np.sqrt(np.real_if_close(val).item())

In [35]:
'''
4体纯态把D trace掉 ρABCD=∣Ψ⟩⟨Ψ∣, ρABC=TrD(ρABCD)
'''
# ===== 4-qubit GHZ state =====
psi4 = np.zeros(16, dtype=complex)
psi4[0]  = 1 / np.sqrt(2)   # |0000>
psi4[15] = 1 / np.sqrt(2)   # |1111>

# density matrix on ABCD
rho4 = np.outer(psi4, psi4.conj())

# trace out D (the 4th qubit, index 3)
rho_ABC = partial_trace(rho4, keep=[0, 1, 2], dims=[2, 2, 2, 2])

print("rho_ABC shape:", rho_ABC.shape)
print(np.round(rho_ABC, 3))
print("trace =", np.trace(rho_ABC))

rho_ABC shape: (8, 8)
[[0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]]
trace = (0.9999999999999998+0j)


In [36]:
# ===== 先算裸 mediator scores 和 J_tot =====
res = mediator_scores_and_tot(H_AB, H_BC, H_AC)

print("=== Mediator scores ===")
print(f"j_AB (mediator = H_AB) = {res['j_AB']:.6f}")
print(f"j_BC (mediator = H_BC) = {res['j_BC']:.6f}")
print(f"j_AC (mediator = H_AC) = {res['j_AC']:.6f}")
print(f"J_tot = {res['J_tot']:.6f}")

# ===== 如果你已经有 rho_ABC，就继续算 J_rho =====
# rho_ABC 应该是 8x8 的三量子比特密度矩阵
J_rho = state_activated_Jrho(rho_ABC, res["A_AB"], res["A_BC"], res["A_AC"])
print(f"J_rho = {J_rho:.6f}")

=== Mediator scores ===
j_AB (mediator = H_AB) = 0.565685
j_BC (mediator = H_BC) = 1.131371
j_AC (mediator = H_AC) = 1.264911
J_tot = 1.788854
J_rho = 0.632456


In [43]:
# ===== benchmark: 4-qubit GHZ =====
psi4 = np.zeros(16, dtype=complex)
psi4[0]  = 1 / np.sqrt(2)
psi4[15] = 1 / np.sqrt(2)

rho4 = np.outer(psi4, psi4.conj())
rho_ABC = partial_trace(rho4, keep=[0,1,2], dims=[2,2,2,2])

print("=== rho_ABC from GHZ4 ===")
print("shape:", rho_ABC.shape)
print("trace:", np.trace(rho_ABC))
print(np.round(rho_ABC, 3))

# ===== mediator associators =====
A_AB = jordan_associator(H_BC, H_AB, H_AC)  # mediator = H_AB
A_BC = jordan_associator(H_AB, H_BC, H_AC)  # mediator = H_BC
A_AC = jordan_associator(H_AB, H_AC, H_BC)  # mediator = H_AC

j_AB = np.linalg.norm(A_AB, 'fro')
j_BC = np.linalg.norm(A_BC, 'fro')
j_AC = np.linalg.norm(A_AC, 'fro')
J_tot = np.sqrt(j_AB**2 + j_BC**2 + j_AC**2)

print("\n=== mediator scores ===")
print("j_AB =", j_AB)
print("j_BC =", j_BC)
print("j_AC =", j_AC)
print("J_tot =", J_tot)

# ===== current J_rho definition =====
M = A_AB.conj().T @ A_AB + A_BC.conj().T @ A_BC + A_AC.conj().T @ A_AC
J_rho = np.sqrt(np.real_if_close(np.trace(rho_ABC @ M)).item())

print("\n=== J_rho ===")
print("J_rho =", J_rho)

print("\n=== check whether M is proportional to identity ===")
print("allclose(M, 2I)?", np.allclose(M, 2*np.eye(8)))
print(np.round(M, 3))

=== rho_ABC from GHZ4 ===
shape: (8, 8)
trace: (0.9999999999999998+0j)
[[0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]]

=== mediator scores ===
j_AB = 0.5656854249492381
j_BC = 1.1313708498984762
j_AC = 1.2649110640673518
J_tot = 1.788854381999832

=== J_rho ===
J_rho = 0.6324555320336759

=== check whether M is proportional to identity ===
allclose(M, 2I)? False
[[0.4+0.j   0. +0.j   0. +0.j   0. +0.j   0. -0.16j 0. +0.j   0. +0.j
  0. +0.j  ]
 [0. +0.j   0.4+0.j   0. +0.j   0. +0.j   0. +0.j   0. +0.16j 

In [42]:
def state_mediator_expectations(rho_ABC, A_AB, A_BC, A_AC):
    m_AB = np.real_if_close(np.trace(rho_ABC @ A_AB)).item()
    m_BC = np.real_if_close(np.trace(rho_ABC @ A_BC)).item()
    m_AC = np.real_if_close(np.trace(rho_ABC @ A_AC)).item()

    J_rho_exp = np.sqrt(np.abs(m_AB)**2 + np.abs(m_BC)**2 + np.abs(m_AC)**2)

    return {
        "m_AB": m_AB,
        "m_BC": m_BC,
        "m_AC": m_AC,
        "J_rho_exp": J_rho_exp,
    }

# A_AB, A_BC, A_AC 是你前面已经算好的三个 mediator associators
state_res = state_mediator_expectations(rho_ABC, A_AB, A_BC, A_AC)

print("=== State-resolved mediator activations ===")
print(f"m_AB = {state_res['m_AB']}")
print(f"m_BC = {state_res['m_BC']}")
print(f"m_AC = {state_res['m_AC']}")
print(f"J_rho_exp = {state_res['J_rho_exp']}")

=== State-resolved mediator activations ===
m_AB = -0.19999999999999996
m_BC = 0.0
m_AC = 0.19999999999999996
J_rho_exp = 0.28284271247461895


In [40]:
'''
Benchmark Cell: 四体纯态 → 三体混态 → 指标输出
input: psi4, output: rho_ABC, I_3, I_3_minus, j_AB, j_BC, j_AC, J_tot, m_AB, m_BC, m_AC, J_rho_exp
for: 
    给定probe能不能区分不同类型的态：product-like, pairwise-dominated, GHZ-like, W-like
    给定probe和TMI的变化是否有关系：J_rho_exp变化，I_3/I_3_minus怎么变化
'''

def I3_of_rhoABC(rho_ABC):
    rho_A  = partial_trace(rho_ABC, keep=[0], dims=[2,2,2])
    rho_B  = partial_trace(rho_ABC, keep=[1], dims=[2,2,2])
    rho_C  = partial_trace(rho_ABC, keep=[2], dims=[2,2,2])
    rho_AB = partial_trace(rho_ABC, keep=[0,1], dims=[2,2,2])
    rho_AC = partial_trace(rho_ABC, keep=[0,2], dims=[2,2,2])
    rho_BC = partial_trace(rho_ABC, keep=[1,2], dims=[2,2,2])

    I3 = (
        von_neumann_entropy(rho_A)
        + von_neumann_entropy(rho_B)
        + von_neumann_entropy(rho_C)
        - von_neumann_entropy(rho_AB)
        - von_neumann_entropy(rho_AC)
        - von_neumann_entropy(rho_BC)
        + von_neumann_entropy(rho_ABC)
    )
    return np.real_if_close(I3).item()

def rhoABC_from_pure4(psi4):
    rho4 = np.outer(psi4, psi4.conj())
    return partial_trace(rho4, keep=[0,1,2], dims=[2,2,2,2])

def analyze_state(name, psi4, H_AB, H_BC, H_AC):
    rho_ABC = rhoABC_from_pure4(psi4)

    A_AB = jordan_associator(H_BC, H_AB, H_AC)
    A_BC = jordan_associator(H_AB, H_BC, H_AC)
    A_AC = jordan_associator(H_AB, H_AC, H_BC)

    j_AB = np.linalg.norm(A_AB, 'fro')
    j_BC = np.linalg.norm(A_BC, 'fro')
    j_AC = np.linalg.norm(A_AC, 'fro')
    J_tot = np.sqrt(j_AB**2 + j_BC**2 + j_AC**2)

    m_AB = np.real_if_close(np.trace(rho_ABC @ A_AB)).item()
    m_BC = np.real_if_close(np.trace(rho_ABC @ A_BC)).item()
    m_AC = np.real_if_close(np.trace(rho_ABC @ A_AC)).item()
    J_rho_exp = np.sqrt(np.abs(m_AB)**2 + np.abs(m_BC)**2 + np.abs(m_AC)**2)

    I3 = I3_of_rhoABC(rho_ABC)
    I3_minus = max(0.0, -I3)

    print(f"\n=== {name} ===")
    print(f"I3 = {I3:.6f}, I3_minus = {I3_minus:.6f}")
    print(f"j_AB = {j_AB:.6f}, j_BC = {j_BC:.6f}, j_AC = {j_AC:.6f}, J_tot = {J_tot:.6f}")
    print(f"m_AB = {m_AB:.6f}, m_BC = {m_BC:.6f}, m_AC = {m_AC:.6f}, J_rho_exp = {J_rho_exp:.6f}")

In [41]:
psi_prod = np.zeros(16, dtype=complex)
psi_prod[0] = 1.0

analyze_state("Product |0000>", psi_prod, H_AB, H_BC, H_AC)

psi_ghz4 = np.zeros(16, dtype=complex)
psi_ghz4[0] = 1/np.sqrt(2)
psi_ghz4[15] = 1/np.sqrt(2)

analyze_state("GHZ4", psi_ghz4, H_AB, H_BC, H_AC)

psi_w4 = np.zeros(16, dtype=complex)
psi_w4[1] = 0.5
psi_w4[2] = 0.5
psi_w4[4] = 0.5
psi_w4[8] = 0.5

analyze_state("W4", psi_w4, H_AB, H_BC, H_AC)

psi_bell = np.zeros(16, dtype=complex)
psi_bell[0] = 1/np.sqrt(2)
psi_bell[12] = 1/np.sqrt(2)

analyze_state("Bell_AB + 00_CD", psi_bell, H_AB, H_BC, H_AC)


=== Product |0000> ===
I3 = 0.000000, I3_minus = 0.000000
j_AB = 0.565685, j_BC = 1.131371, j_AC = 1.264911, J_tot = 1.788854
m_AB = -0.200000, m_BC = 0.000000, m_AC = 0.200000, J_rho_exp = 0.282843

=== GHZ4 ===
I3 = 1.000000, I3_minus = 0.000000
j_AB = 0.565685, j_BC = 1.131371, j_AC = 1.264911, J_tot = 1.788854
m_AB = -0.200000, m_BC = 0.000000, m_AC = 0.200000, J_rho_exp = 0.282843

=== W4 ===
I3 = 0.245112, I3_minus = 0.000000
j_AB = 0.565685, j_BC = 1.131371, j_AC = 1.264911, J_tot = 1.788854
m_AB = 0.000000, m_BC = 0.000000, m_AC = 0.000000, J_rho_exp = 0.000000

=== Bell_AB + 00_CD ===
I3 = 0.000000, I3_minus = 0.000000
j_AB = 0.565685, j_BC = 1.131371, j_AC = 1.264911, J_tot = 1.788854
m_AB = 0.000000, m_BC = 0.000000, m_AC = 0.000000, J_rho_exp = 0.000000


In [44]:
from itertools import product

paulis = {"X": sx, "Y": sy, "Z": sz}

# --- generate all single-string edge probes ---
ops_AB = []
ops_BC = []
ops_AC = []

for a, b in product(paulis.keys(), repeat=2):
    ops_AB.append((f"{a}{b}I", np.kron(np.kron(paulis[a], paulis[b]), I2)))
    ops_BC.append((f"I{a}{b}", np.kron(np.kron(I2, paulis[a]), paulis[b])))
    ops_AC.append((f"{a}I{b}", np.kron(np.kron(paulis[a], I2), paulis[b])))

def rhoABC_from_pure4(psi4):
    rho4 = np.outer(psi4, psi4.conj())
    return partial_trace(rho4, keep=[0,1,2], dims=[2,2,2,2])

def Jrho_exp_for_state(rho_ABC, H_AB, H_BC, H_AC):
    A_AB = jordan_associator(H_BC, H_AB, H_AC)  # mediator = H_AB
    A_BC = jordan_associator(H_AB, H_BC, H_AC)  # mediator = H_BC
    A_AC = jordan_associator(H_AB, H_AC, H_BC)  # mediator = H_AC

    m_AB = np.real_if_close(np.trace(rho_ABC @ A_AB)).item()
    m_BC = np.real_if_close(np.trace(rho_ABC @ A_BC)).item()
    m_AC = np.real_if_close(np.trace(rho_ABC @ A_AC)).item()

    J_rho_exp = np.sqrt(np.abs(m_AB)**2 + np.abs(m_BC)**2 + np.abs(m_AC)**2)
    return J_rho_exp, (m_AB, m_BC, m_AC), (A_AB, A_BC, A_AC)

# --- benchmark states ---
psi_prod = np.zeros(16, dtype=complex)
psi_prod[0] = 1.0

psi_ghz4 = np.zeros(16, dtype=complex)
psi_ghz4[0] = 1/np.sqrt(2)
psi_ghz4[15] = 1/np.sqrt(2)

rho_prod = rhoABC_from_pure4(psi_prod)
rho_ghz4 = rhoABC_from_pure4(psi_ghz4)

results = []

for name_ab, H_AB in ops_AB:
    for name_bc, H_BC in ops_BC:
        for name_ac, H_AC in ops_AC:
            J_prod, m_prod, A_ops = Jrho_exp_for_state(rho_prod, H_AB, H_BC, H_AC)
            J_ghz,  m_ghz,  _     = Jrho_exp_for_state(rho_ghz4, H_AB, H_BC, H_AC)

            # 要求：至少有非零 HOI channel，并且能区分 Product vs GHZ4
            score = abs(J_prod - J_ghz)

            if score > 1e-8:
                results.append({
                    "AB": name_ab,
                    "BC": name_bc,
                    "AC": name_ac,
                    "J_prod": J_prod,
                    "J_ghz": J_ghz,
                    "score": score,
                    "m_prod": m_prod,
                    "m_ghz": m_ghz,
                })

results = sorted(results, key=lambda x: -x["score"])

print(f"Found {len(results)} distinguishing probe families.\n")
for r in results[:20]:
    print(r["AB"], r["BC"], r["AC"],
          " | J_prod =", round(r["J_prod"], 6),
          " | J_ghz =", round(r["J_ghz"], 6),
          " | score =", round(r["score"], 6),
          " | m_prod =", tuple(round(float(np.real(v)), 6) for v in r["m_prod"]),
          " | m_ghz =", tuple(round(float(np.real(v)), 6) for v in r["m_ghz"]))

Found 0 distinguishing probe families.

